In [17]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from langchain_core.runnables import RunnableBranch, RunnableLambda
from pydantic import BaseModel, Field
from typing import Literal

In [18]:
model = ChatOpenAI(model = 'gpt-3.5-turbo')

In [19]:
parser = StrOutputParser()

In [20]:
class Sentiment_Analysis(BaseModel):
    sentiment:Literal['Positive', 'Negative', 'Neutral'] = Field(description="The sentiment of this field can be positive, negative or neutral")

In [21]:
pyd_parser = PydanticOutputParser(pydantic_object=Sentiment_Analysis)

In [22]:
template = PromptTemplate(template = "Give the sentiment of the following feedback \n {feedback}\n {fmt}",
                          input_variables=['feedback'],
                          partial_variables={'fmt':pyd_parser.get_format_instructions()}
                        )

In [23]:
chain = template | model | pyd_parser

In [24]:
result = chain.invoke({'feedback': "I am happy with the feedback"})
print(result)

sentiment='Positive'


In [25]:
print(result.sentiment)

Positive


Branching

In [26]:
template_positive = PromptTemplate(template='Ask customer for review on Mobile app as well because customer {response}',
                                   input_variables=['response'])

In [27]:
template_negative = PromptTemplate(template = "Reply customer politely and tell customer that custoemr care will reply respond to you because customer {response}",
                                   input_variables=['response'])

In [28]:
branch_chain = RunnableBranch(
    (lambda x: x.sentiment == 'Positive', template_positive | model | parser),
    (lambda x: x.sentiment == 'Negative', template_negative | model | parser),
    RunnableLambda(lambda x: 'Wrong Output')
)

In [29]:
final_chain = chain | branch_chain
result = final_chain.invoke({'feedback': 'I am not happy with the product'})

In [30]:
print(result)

Dear customer,

Thank you for reaching out to us. We are sorry to hear that you are experiencing a negative sentiment regarding our service. Our customer care team will respond to you as soon as possible to address your concerns.

Please do not hesitate to contact us if you have any further questions or feedback.

Best regards,
[Your Name]
[Company Name]
